In [1]:
from pathlib import Path
import pandas as pd

# 1. BASE DIRECTORY 

BASE = Path.home() / "Desktop" / "LCS_eligibility_Ireland"

CODE_DIR = BASE / "code"
RAW_DIR = BASE / "data_raw"
PROC_DIR = BASE / "data_processed"
OUT_DIR = BASE / "output"

print("Using base directory:", BASE)

Using base directory: C:\Users\tatianabezdenezhnykh\Desktop\LCS_eligibility_Ireland


In [2]:


# Define the file path
file_path = RAW_DIR / "restructured_smoking_population_data.csv"

data = pd.read_csv(file_path)


In [3]:
data.head()


,Statistic Label,Census Year,Sex,Smoking Tobacco Products,Age Group,County,UNIT,VALUE
0,Population,2022,Both Sexes,Smoke daily,All Ages,State,Number,449703.0
1,Population,2022,Both Sexes,Smoke daily,0 - 14 years,State,Number,572.0
2,Population,2022,Both Sexes,Smoke daily,15-19 years,State,Number,8459.0
3,Population,2022,Both Sexes,Smoke daily,20-24 years,State,Number,30998.0
4,Population,2022,Both Sexes,Smoke daily,25-29 years,State,Number,39737.0


In [4]:
# Define the age group mapping
age_group_mapping = {
    '0 - 14 years': '0-14',
    '15-19 years': '15-19',
    '20-24 years': '20-24',
    '25-29 years': '25-29',
    '30-34 years': '30-34',
    '35-39 years': '35-39',
    '40-44 years': '40-44',
    '45-49 years': '45-49',
    '50-54 years': '50-54',
    '55-59 years': '55-59',
    '60-64 years': '60-64',
    '65-69 years': '65-69',
    '70-74 years': '70-74',
    '75-79 years': '75-79',
    '80-84 years': '80-84',
    '85-89 years': '85-89',
    '95 years and over': '90 and over' 
}
# Map age groups in the dataset to the specified categories
data['age_group'] = data['Age Group'].map(age_group_mapping)



In [5]:
# Display unique values for each column except 'VALUE'
unique_values = {col: data[col].unique() for col in data.columns if col != 'VALUE'}

# Displaying the unique values for each column
unique_values

{'Statistic Label': array(['Population'], dtype=object),
 'Census Year': array([2022], dtype=int64),
 'Sex': array(['Both Sexes', 'Male', 'Female'], dtype=object),
 'Smoking Tobacco Products': array(['Smoke daily', 'Smoke occasionally', 'Dont smoke - gave up',
        'Never smoked', 'Not stated'], dtype=object),
 'Age Group': array(['All Ages', '0 - 14 years', '15-19 years', '20-24 years',
        '25-29 years', '30-34 years', '35-39 years', '40-44 years',
        '45-49 years', '50-54 years', '55-59 years', '60-64 years',
        '65-69 years', '70-74 years', '75-79 years', '80-84 years',
        '85-89 years', '95 years and over'], dtype=object),
 'County': array(['State', 'Carlow', 'Dublin City', 'Dun Laoghaire-Rathdown',
        'Fingal', 'South Dublin', 'Kildare', 'Kilkenny', 'Laois',
        'Longford', 'Louth', 'Meath', 'Offaly', 'Westmeath', 'Wexford',
        'Wicklow', 'Clare', 'Cork City', 'Cork County', 'Kerry',
        'Limerick City and County', 'Tipperary',
        'Wat

In [6]:
# Dropping observations based on specified conditions
filtered_data = data[~(
    (data['Sex'] == 'Both Sexes') |
    (data['County'] == 'State') |
    (data['Age Group'] == 'All Ages')
)]

In [7]:
# Display unique values for each column except 'VALUE'
unique_values = {col: filtered_data[col].unique() for col in data.columns if col != 'VALUE'}

# Displaying the unique values for each column
unique_values

{'Statistic Label': array(['Population'], dtype=object),
 'Census Year': array([2022], dtype=int64),
 'Sex': array(['Male', 'Female'], dtype=object),
 'Smoking Tobacco Products': array(['Smoke daily', 'Smoke occasionally', 'Dont smoke - gave up',
        'Never smoked', 'Not stated'], dtype=object),
 'Age Group': array(['0 - 14 years', '15-19 years', '20-24 years', '25-29 years',
        '30-34 years', '35-39 years', '40-44 years', '45-49 years',
        '50-54 years', '55-59 years', '60-64 years', '65-69 years',
        '70-74 years', '75-79 years', '80-84 years', '85-89 years',
        '95 years and over'], dtype=object),
 'County': array(['Carlow', 'Dublin City', 'Dun Laoghaire-Rathdown', 'Fingal',
        'South Dublin', 'Kildare', 'Kilkenny', 'Laois', 'Longford',
        'Louth', 'Meath', 'Offaly', 'Westmeath', 'Wexford', 'Wicklow',
        'Clare', 'Cork City', 'Cork County', 'Kerry',
        'Limerick City and County', 'Tipperary',
        'Waterford City and County', 'Galway Ci

In [8]:
duplicates_check = data.duplicated(subset=['County', 'Sex', 'Smoking Tobacco Products', 'age_group'])
duplicates_check.any()

False

In [9]:
# Filter for age groups between 50 and 79
age_groups_50_79 = [f"{age}-{age+4}" for age in range(50, 80, 5)]
df_50_79 = filtered_data[filtered_data["age_group"].isin(age_groups_50_79)]

# Calculate total population in each age group
total_by_age_group = df_50_79.groupby("age_group")["VALUE"].sum().reset_index()

# Calculate total population aged 50–79
total_50_79 = total_by_age_group["VALUE"].sum()


total_by_age_group.head()

,age_group,VALUE
0,50-54,340003.0
1,55-59,307165.0
2,60-64,272670.0
3,65-69,238144.0
4,70-74,202884.0


In [10]:
# Map the statuses to simpler terms: Smokers, Quitters, and Never-Smokers
filtered_data['Smoking Tobacco Products'] = filtered_data['Smoking Tobacco Products'].replace({
    'Smoke daily': 'Smokers',
    'Smoke occasionally': 'Smokers',
    'Dont smoke - gave up': 'Quitters',
    'Never smoked': 'Never-Smokers',
    'Not stated': 'Not Stated'
})



C:\Users\tatianabezdenezhnykh\AppData\Local\Temp\ipykernel_24972\2303748828.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_data['Smoking Tobacco Products'] = filtered_data['Smoking Tobacco Products'].replace({


In [11]:
# Dictionary mapping NUTS3 (counties) to NUTS2 regions
# Updated dictionary mapping NUTS3 (counties) to NUTS2 regions
nuts3_to_nuts2 = {
    # Northern & Western NUTS2 Region
    'Cavan': 'Border',
    'Donegal': 'Border',
    'Leitrim': 'Border',
    'Monaghan': 'Border',
    'Sligo': 'Border',
    'Galway City': 'West',
    'Galway County': 'West',
    'Mayo': 'West',
    'Roscommon': 'West',

    # Southern NUTS2 Region
    'Clare': 'Mid-West',
    'Limerick City and County': 'Mid-West',  # fixed spelling
    'Tipperary': 'Mid-West',
    'Carlow': 'South-East',
    'Kilkenny': 'South-East',
    'Waterford City and County': 'South-East',  # fixed spelling
    'Wexford': 'South-East',
    'Cork City': 'South-West',
    'Cork County': 'South-West',
    'Kerry': 'South-West',

    # Eastern & Midland NUTS2 Region
    'Dublin City': 'Dublin',
    'Dun Laoghaire-Rathdown': 'Dublin',  # fixed spelling
    'Fingal': 'Dublin',
    'South Dublin': 'Dublin',
    'Kildare': 'Mid-East',
    'Meath': 'Mid-East',
    'Wicklow': 'Mid-East',
    'Louth': 'Mid-East',
    'Laois': 'Midland',
    'Longford': 'Midland',
    'Offaly': 'Midland',
    'Westmeath': 'Midland'
}



In [12]:
# Add a new column 'NUTS2 Region' based on the 'CSO Local Electoral Areas' (NUTS3 regions)
filtered_data['region'] = filtered_data['County'].map(nuts3_to_nuts2)

# Display the first few rows to confirm the changes
filtered_data.head()


C:\Users\tatianabezdenezhnykh\AppData\Local\Temp\ipykernel_24972\1687520095.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_data['region'] = filtered_data['County'].map(nuts3_to_nuts2)


,Statistic Label,Census Year,Sex,Smoking Tobacco Products,Age Group,County,UNIT,VALUE,age_group,region
2899,Population,2022,Male,Smokers,0 - 14 years,Carlow,Number,NaN,0-14,South-East
2900,Population,2022,Male,Smokers,15-19 years,Carlow,Number,69.0,15-19,South-East
2901,Population,2022,Male,Smokers,20-24 years,Carlow,Number,270.0,20-24,South-East
2902,Population,2022,Male,Smokers,25-29 years,Carlow,Number,348.0,25-29,South-East
2903,Population,2022,Male,Smokers,30-34 years,Carlow,Number,325.0,30-34,South-East


In [13]:
# Identify counties with missing NUTS2 region mapping
missing_region = filtered_data[filtered_data['region'].isna()]['County'].unique()
print("Counties with missing NUTS2 mapping:", missing_region)

Counties with missing NUTS2 mapping: []


In [14]:
# Filter for age groups between 50 and 79
age_groups_50_79 = [f"{age}-{age+4}" for age in range(50, 80, 5)]
df_50_79 = filtered_data[filtered_data["age_group"].isin(age_groups_50_79)]

# Calculate total population in each age group
total_by_age_group = df_50_79.groupby("age_group")["VALUE"].sum().reset_index()

# Calculate total population aged 50–79
total_50_79 = total_by_age_group["VALUE"].sum()


total_by_age_group.head()
#print(total_50_79)  ###total number CSO projections 1,528,640

,age_group,VALUE
0,50-54,340003.0
1,55-59,307165.0
2,60-64,272670.0
3,65-69,238144.0
4,70-74,202884.0


In [15]:
# Group by region, gender, smoking status, and mapped age group, and sum the population counts
filtered_data = filtered_data.groupby(
    ['region', 'Sex', 'Smoking Tobacco Products', 'age_group']
)['VALUE'].sum().reset_index()

# Filter for age groups between 50 and 79
age_groups_50_79 = [f"{age}-{age+4}" for age in range(50, 80, 5)]
df_50_79 = filtered_data[filtered_data["age_group"].isin(age_groups_50_79)]

# Calculate total population in each age group
total_by_age_group = df_50_79.groupby("age_group")["VALUE"].sum().reset_index()

# Calculate total population aged 50–79
total_50_79 = total_by_age_group["VALUE"].sum()


total_by_age_group.head()

,age_group,VALUE
0,50-54,340003.0
1,55-59,307165.0
2,60-64,272670.0
3,65-69,238144.0
4,70-74,202884.0


In [16]:
# Renaming columns for better clarity and consistency
filtered_data = filtered_data.rename(columns={
    'Sex': 'gender',
    'Smoking Tobacco Products': 'smoking_status',
    'Mapped Age Group': 'age_group',
    'VALUE': 'population'
})

filtered_data.head()

,region,gender,smoking_status,age_group,population
0,Border,Female,Never-Smokers,0-14,38452.0
1,Border,Female,Never-Smokers,15-19,12065.0
2,Border,Female,Never-Smokers,20-24,7365.0
3,Border,Female,Never-Smokers,25-29,5851.0
4,Border,Female,Never-Smokers,30-34,7118.0


In [17]:
# Convert 'gender', 'smoking_status', and 'age_group' in the smoking dataset to categorical codes

# Gender encoding
filtered_data['gender_code'] = filtered_data['gender'].astype('category').cat.codes

# Smoking status encoding
filtered_data['smoking_status_code'] = filtered_data['smoking_status'].astype('category').cat.codes

# Age group encoding
filtered_data['age_group_code'] = filtered_data['age_group'].astype('category').cat.codes

# Display the first few rows to confirm the encoding
filtered_data.head(40)


,region,gender,smoking_status,age_group,population,gender_code,smoking_status_code,age_group_code
0,Border,Female,Never-Smokers,0-14,38452.0,0,0,0
1,Border,Female,Never-Smokers,15-19,12065.0,0,0,1
2,Border,Female,Never-Smokers,20-24,7365.0,0,0,2
3,Border,Female,Never-Smokers,25-29,5851.0,0,0,3
4,Border,Female,Never-Smokers,30-34,7118.0,0,0,4
5,Border,Female,Never-Smokers,35-39,8158.0,0,0,5
6,Border,Female,Never-Smokers,40-44,8648.0,0,0,6
7,Border,Female,Never-Smokers,45-49,8107.0,0,0,7
8,Border,Female,Never-Smokers,50-54,7353.0,0,0,8
9,Border,Female,Never-Smokers,55-59,7272.0,0,0,9


In [18]:
# Filter for age groups between 50 and 79
age_groups_50_79 = [f"{age}-{age+4}" for age in range(50, 80, 5)]
df_50_79 = filtered_data[filtered_data["age_group"].isin(age_groups_50_79)]

# Calculate total population in each age group
total_by_age_group = df_50_79.groupby("age_group")["population"].sum().reset_index()

# Calculate total population aged 50–79
total_50_79 = total_by_age_group["population"].sum()

total_by_age_group.head()
#print(total_50_79)  ###total number CSO projections 1,528,640

,age_group,population
0,50-54,340003.0
1,55-59,307165.0
2,60-64,272670.0
3,65-69,238144.0
4,70-74,202884.0


In [19]:
# Group by relevant categories and sum the population
grouped_data = filtered_data.groupby(['region', 'age_group', 'gender', 'smoking_status']).agg({'population': 'sum'}).reset_index()

# Pivot the data to separate smoking statuses into columns
pivoted_data = grouped_data.pivot_table(
    index=['region', 'age_group', 'gender'],
    columns='smoking_status',
    values='population',
    fill_value=0
).reset_index()

# Calculate total population and percentages for each smoking status
pivoted_data['Total Population'] = pivoted_data.sum(axis=1, numeric_only=True)
pivoted_data['Smokers (%)'] = (pivoted_data.get('Smokers', 0) / pivoted_data['Total Population']) 
pivoted_data['Quitters (%)'] = (pivoted_data.get('Quitters', 0) / pivoted_data['Total Population']) 
pivoted_data['Never-Smokers (%)'] = (pivoted_data.get('Never-Smokers', 0) / pivoted_data['Total Population']) 

# Select and rename columns 
final_data = pivoted_data.rename(
    columns={'Total Population': 'population', 'Smokers (%)': 'smokers', 'Quitters (%)': 'quitters', 'Never-Smokers (%)': 'never_smokers'}
)[['region', 'age_group', 'gender', 'population', 'smokers', 'quitters', 'never_smokers']]

final_data = final_data[~final_data["age_group"].isin(["80-84","85-89", "90 and over", "0 - 14"])].copy()

print(final_data.head(10))

# Print total population for filtered age groups (55-64, 65-74) in merged_data
total_population_filtered = final_data["population"].sum()

print(f"Total population in age groups 55-64 and 65-74: {total_population_filtered:.0f}")


smoking_status  region age_group  gender  population   smokers  quitters  \
0               Border      0-14  Female     42214.0  0.000000  0.007604   
1               Border      0-14    Male     44302.0  0.000271  0.004469   
2               Border     15-19  Female     13778.0  0.045580  0.027435   
3               Border     15-19    Male     14539.0  0.072770  0.033978   
4               Border     20-24  Female     10332.0  0.164731  0.063105   
5               Border     20-24    Male     11169.0  0.240666  0.072254   
6               Border     25-29  Female      9483.0  0.204998  0.122008   
7               Border     25-29    Male      9705.0  0.311901  0.116950   
8               Border     30-34  Female     12056.0  0.175348  0.174934   
9               Border     30-34    Male     11000.0  0.280455  0.157364   

smoking_status  never_smokers  
0                    0.910883  
1                    0.916505  
2                    0.875671  
3                    0.845313  
4  


Total population in age groups 55-64 and 65-74: 4966136


In [20]:
file_path = PROC_DIR / "cleaned_smoking_data.csv"
final_data.to_csv(file_path, index=False)

final_data.head()

smoking_status,region,age_group,gender,population,smokers,quitters,never_smokers
0,Border,0-14,Female,42214.0,0.000000,0.007604,0.910883
1,Border,0-14,Male,44302.0,0.000271,0.004469,0.916505
2,Border,15-19,Female,13778.0,0.045580,0.027435,0.875671
3,Border,15-19,Male,14539.0,0.072770,0.033978,0.845313
4,Border,20-24,Female,10332.0,0.164731,0.063105,0.712834


In [21]:
# Filter for age groups between 50 and 79
age_groups_50_79 = [f"{age}-{age+4}" for age in range(50, 80, 5)]
df_50_79 = final_data[final_data["age_group"].isin(age_groups_50_79)]

# Calculate total population in each age group
total_by_age_group = df_50_79.groupby("age_group")["population"].sum().reset_index()

# Calculate total population aged 50–79
total_50_79 = total_by_age_group["population"].sum()


print(total_50_79)  ###total number CSO projections 1,528,640

1515126.0


In [22]:
total_by_age_group.head()

,age_group,population
0,50-54,340003.0
1,55-59,307165.0
2,60-64,272670.0
3,65-69,238144.0
4,70-74,202884.0


In [23]:
# Display unique values for each column except 'VALUE'
unique_values = {col: final_data[col].unique() for col in final_data.columns if col != 'VALUE'}

# Displaying the unique values for each column
unique_values


{'region': array(['Border', 'Dublin', 'Mid-East', 'Mid-West', 'Midland',
        'South-East', 'South-West', 'West'], dtype=object),
 'age_group': array(['0-14', '15-19', '20-24', '25-29', '30-34', '35-39', '40-44',
        '45-49', '50-54', '55-59', '60-64', '65-69', '70-74', '75-79'],
       dtype=object),
 'gender': array(['Female', 'Male'], dtype=object),
 'population': array([ 42214.,  44302.,  13778.,  14539.,  10332.,  11169.,   9483.,
          9705.,  12056.,  11000.,  14833.,  13626.,  16149.,  15276.,
         14972.,  14699.,  14089.,  14279.,  13361.,  13139.,  12304.,
         11979.,  10975.,  11107.,   9411.,   9538.,   7210.,   7111.,
        131090., 137489.,  43685.,  44304.,  50643.,  48921.,  55409.,
         54570.,  59696.,  57155.,  61366.,  58328.,  61215.,  59256.,
         51550.,  50718.,  45307.,  44280.,  40456.,  37879.,  35683.,
         33126.,  30885.,  27590.,  26644.,  23308.,  20912.,  17460.,
         80057.,  84467.,  26525.,  27930.,  20719.,  22

In [24]:


final_data_ag = (
    final_data
    .groupby(["age_group", "gender"], as_index=False)  # Group by age_group and gender
    .agg({
        "population": "sum",  # Sum populations
        "smokers": lambda x: (x * final_data.loc[x.index, "population"]).sum() / final_data.loc[x.index, "population"].sum(),
        "quitters": lambda x: (x * final_data.loc[x.index, "population"]).sum() / final_data.loc[x.index, "population"].sum(),
        "never_smokers": lambda x: (x * final_data.loc[x.index, "population"]).sum() / final_data.loc[x.index, "population"].sum()
    })
)

# Inspect the final aggregated DataFrame
print(final_data_ag.head())

# Print total population for filtered age groups (55-64, 65-74) in merged_data
total_population_filtered = final_data_ag["population"].sum()

print(f"Total population in age groups 55-64 and 65-74: {total_population_filtered:.0f}")

smoking_status age_group  gender  population   smokers  quitters  \
0                   0-14  Female    493154.0  0.000047  0.005217   
1                   0-14    Male    517157.0  0.000124  0.006576   
2                  15-19  Female    165286.0  0.050228  0.025029   
3                  15-19    Male    172342.0  0.070058  0.029691   
4                  20-24  Female    151697.0  0.170452  0.063488   

smoking_status  never_smokers  
0                    0.897207  
1                    0.894177  
2                    0.857550  
3                    0.831440  
4                    0.692407  
Total population in age groups 55-64 and 65-74: 4966136


In [25]:


file_path = PROC_DIR / "cleaned_smoking_data_ag.csv"
final_data_ag.to_csv(file_path, index=False)